In [3]:
"""
PIPELINE STAGE: Multimodal Data Integration & Temporal Broadcasting
TIMELINE: 2020 - 2024 (Longitudinal Study)
INPUTS: 
    - Master Energy Database: 07_final_energy_consolidation.xlsx (Extracted from unstructured sources)
    - Demographic Dataset: 08_a_cleaned_population.xlsx (Parsed from non-standard horizontal CSV)
OUTPUT: 09_energy_and_population.xlsx
LIBRARIES: pandas, os

1. RESEARCH CORE: CROSS-FORMAT DATA SYNTHESIS
    The primary focus of this stage is the consolidation of data derived from heterogeneous 
    and non-standard formats. While the energy data was previously mined from unstructured 
    Word/Excel files using LLM-assisted workflows, the demographic data was extracted 
    from a 'Horizontal Sequential CSV' format. This stage synchronizes these disparate 
    streams into a unified relational database.

2. TECHNICAL LOGIC: TEMPORAL BROADCASTING
    - Granularity Alignment: The demographic data (annual) is integrated into the 
      energy dataset (monthly) through 'temporal broadcasting'. The singular yearly 
      population value is replicated across 12 monthly temporal units for each province.
    - Composite Key Integration: To eliminate nomenclature risks (orthographic variations 
      in province names), a relational 'Left Join' is executed using [Plate] and [Year] 
      as the composite primary keys.

3. SCHEMA ARCHITECTURE & INTEGRITY:
    - Preservation: The original 10-column energy schema (Plate, Province, Year, Month, 
      Consumption, and sectors) is maintained, with 'Population' appended as the 
      final feature column.
    - Type Enforcement: Numerical integrity is ensured by casting all join-keys as 
      Integers, preventing data leakage or null-value propagation during the merge.

4. METHODOLOGICAL SIGNIFICANCE:
    By anchoring annual demographic denominators to monthly sectoral energy consumption, 
    this process enables high-resolution 'Per-Capita' modeling across the 2020-2024 
    timeline, serving as the backbone for subsequent socioeconomic trend analysis.
"""



import pandas as pd
import os

def enerji_ve_nufus_birlestir():
    # 1. Dosya Yolları
    # Ana enerji dosyası (processed_data/steps içindeki son halini referans alıyoruz)
    ENERGY_PATH = os.path.join("..", "..", "processed_data", "steps","07_final_energy_consolidation.xlsx")
    # Yeni temizlediğimiz nüfus dosyası
    POP_PATH = os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")
    # Final çıktı yolu
    FINAL_OUT = os.path.join("..", "..", "processed_data", "steps", "09_energy_and_population.xlsx")
    
    try:
        # 2. Dosyaları Oku
        print("Tablolar okunuyor...")
        df_energy = pd.read_excel(ENERGY_PATH)
        df_pop = pd.read_excel(POP_PATH)

        # Sütun isimlerindeki olası boşlukları temizleyelim
        df_energy.columns = [c.strip() for c in df_energy.columns]
        df_pop.columns = [c.strip() for c in df_pop.columns]

        # 3. Veri Tiplerini Eşitle (Kritik: Plate ve Year her iki tabloda da integer olmalı)
        df_energy['Plate'] = df_energy['Plate'].astype(int)
        df_energy['Year'] = df_energy['Year'].astype(int)
        df_pop['Plate'] = df_pop['Plate'].astype(int)
        df_pop['Year'] = df_pop['Year'].astype(int)

        # 4. Birleştirme (Left Join)
        # Nüfus tablosundan sadece eşleşme anahtarları ve Population sütununu alıyoruz
        print("Plate ve Year bazında nüfus verileri entegre ediliyor...")
        final_df = pd.merge(
            df_energy, 
            df_pop[['Plate', 'Year', 'Population']], 
            on=['Plate', 'Year'], 
            how='left'
        )

        # 5. Son Kontroller
        # Eğer bazı satırlarda nüfus boş kaldıysa uyarı ver
        missing_count = final_df['Population'].isna().sum()
        if missing_count > 0:
            print(f"UYARI: {missing_count} satırda nüfus verisi eşleşmedi! Lütfen eksik yılları/plakaları kontrol edin.")

        # 6. Kaydet
        os.makedirs(os.path.dirname(FINAL_OUT), exist_ok=True)
        final_df.to_excel(FINAL_OUT, index=False)
        
        print("\n" + "="*40)
        print(f"İŞLEM BAŞARIYLA TAMAMLANDI")
        print(f"Yeni Sütun: Population (En sona eklendi)")
        print(f"Toplam Satır: {len(final_df)}")
        print(f"Dosya: {FINAL_OUT}")
        print("="*40)

    except Exception as e:
        print(f"HATA OLUŞTU: {e}")

if __name__ == "__main__":
    enerji_ve_nufus_birlestir()

Tablolar okunuyor...
Plate ve Year bazında nüfus verileri entegre ediliyor...

İŞLEM BAŞARIYLA TAMAMLANDI
Yeni Sütun: Population (En sona eklendi)
Toplam Satır: 4860
Dosya: ..\..\processed_data\steps\09_energy_and_population.xlsx
